# GloVe (mean pooling, concatenated) + unidirectional LSTM + MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** Same as `rnn_basic.ipynb`, only the recurrent layer changes:

- same preprocessing and GloVe sentence vectors (mean pooling, `glove-wiki-gigaword-300`)
- the concatenated 1200 dim vector is read as a sequence of 4 time steps O1 -> O2 -> H1 -> H2, 300 dims each
- one layer, unidirectional LSTM (`nn.LSTM`), its last hidden state summarizes the whole example, the classifier gets the hidden state h, the cell state c is not used
- MLP head: linear -> ReLU -> linear -> 2 logits
- softmax over the 2 logits -> probability for H1 and H2, the answer is the more probable hypothesis

In [1]:
%pip install -q datasets gensim lightning nltk python-dotenv wandb


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from collections import Counter
from pathlib import Path

import gensim.downloader as api
import lightning as L
import nltk
import numpy as np
import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader, TensorDataset

nltk.download("punkt_tab", quiet=True)

True

In [3]:
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
print(f"environment loaded from: {dotenv_path or '<no .env found, using the defaults below>'}")

WANDB_BASE_URL = os.getenv("WANDB_BASE_URL", "https://api.wandb.ai")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "cmedi32-hochschule-luzern")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nlp-project-01-abductive-reasoning")

os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
config = {
    "dataset": "amarfurt/art-nlp-hs26",
    "tokenizer": "nltk.word_tokenize",
    "lowercase": True,
    "remove_punctuation": True,
    "remove_stopwords": False,
    "normalization": "none",
    "truncation": "none",
    "embedding": "glove-wiki-gigaword-300",
    "sentence_representation": "mean",
    "example_representation": "sequence",  # O1, O2, H1, H2 -> 4 rnn time steps
    "encoder": "lstm",  # rnn, gru, lstm
    "bidirectional": False,
    "num_layers": 1,
    "rnn_hidden_dim": 256,
    "classifier": "mlp",  # linear -> relu -> linear
    "hidden_dim": 256,
    "learning_rate": 1e-4,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="glove-concatenate-lstm-unidirectional",
    config=config,
)

In [5]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


In [6]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token.lower() for token in tokens if any(character.isalnum() for character in token)]

In [7]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [8]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

In [9]:
wv = api.load(config["embedding"])
print(f"vocabulary: {len(wv.index_to_key)} words, {wv.vector_size} dimensions")

vocabulary: 400000 words, 300 dimensions


In [10]:
oov_counts = Counter()
embed_stats = Counter()

def embed(tokens):
    vectors = []
    for token in tokens:
        embed_stats["tokens"] += 1
        if token in wv:
            vectors.append(wv[token])
        else:
            embed_stats["oov"] += 1
            oov_counts[token] += 1
    if not vectors:
        embed_stats["empty_sentences"] += 1
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

In [11]:
def embed_example(example):
    vectors = [embed(example[field]) for field in TEXT_FIELDS]
    return np.concatenate(vectors) # this simple change in combination with much lower OOV rate made a huge difference.

In [12]:
X_train = np.stack([embed_example(example) for example in train_tokens])
y_train = np.array(train_tokens["target"])
X_valid = np.stack([embed_example(example) for example in valid_tokens])
y_valid = np.array(valid_tokens["target"])
X_test = np.stack([embed_example(example) for example in test_tokens])
y_test = np.array(test_tokens["target"])

### DataLoaders - Takes care of Training Batching and Loading

In [13]:
L.seed_everything(config["seed"])

def make_loader(X, y, shuffle):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
valid_loader = make_loader(X_valid, y_valid, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

Seed set to 42


### LightningModule Model

In [14]:
class LSTMClassifier(L.LightningModule):

    def __init__(self, input_dim, rnn_hidden_dim, hidden_dim, output_dim, num_layers, bidirectional, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        self.rnn = nn.LSTM(input_dim, rnn_hidden_dim, num_layers=num_layers, bidirectional=bidirectional, batch_first=True)
        num_directions = 2 if bidirectional else 1
        self.classifier = nn.Sequential(
            nn.Linear(rnn_hidden_dim * num_directions, hidden_dim),  # bidirectional: forward and backward state concatenated
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        # (batch, 4 * 300) -> (batch, 4, 300): one time step per sentence O1, O2, H1, H2
        sequence = x.view(x.size(0), -1, self.hparams.input_dim)
        _, (hidden, _) = self.rnn(sequence)  # hidden h: (num_layers * num_directions, batch, rnn_hidden_dim), cell state c unused
        if self.hparams.bidirectional:
            features = torch.cat([hidden[-2], hidden[-1]], dim=1)  # @todo special about this SYSTEM!! last layer: forward state after H2, backward state after O1
        else:
            features = hidden[-1]  # last layer: state after reading H2
        return self.classifier(features)  # logits: (batch, 2)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def shared_step(self, batch, split):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)  # CrossEntropyLoss applies the softmax itself, so it gets the raw logits
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return torch.softmax(self(x), dim=1)  # (batch, 2): p(hypothesis_1), p(hypothesis_2), each row sums to 1

In [15]:
WEIGHTS_DIR = Path("../../../models/project_1/glove_concatenate_lstm_unidirectional/weights")

model = LSTMClassifier(
    input_dim=wv.vector_size,  # one time step = one 300 dim sentence vector
    rnn_hidden_dim=config["rnn_hidden_dim"],
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    num_layers=config["num_layers"],
    bidirectional=config["bidirectional"],
    learning_rate=config["learning_rate"],
)

early_stopping = EarlyStopping(monitor="loss/valid", mode="min", patience=config["patience"], verbose=True)
checkpoint = ModelCheckpoint(dirpath=WEIGHTS_DIR, filename="best-{epoch}", monitor="loss/valid", mode="min", save_top_k=1)

trainer = L.Trainer(
    max_epochs=config["max_epochs"],
    accelerator="auto",
    devices=1,
    logger=WandbLogger(),  # reuses the run from wandb.init
    callbacks=[early_stopping, checkpoint],
)
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ rnn        │ LSTM             │  571 K │ train │     0 │
│ 1 │ classifier │ Sequential       │ 66.3 K │ train │     0 │
│ 2 │ loss_fn    │ CrossEntropyLoss │      0 │ train │     0 │
└───┴────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 637 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 637 K                                                                                                
Total estimated model params size (MB): 2.551                                                                      
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.693
Metric loss/valid improved by 0.000 >= min_delta = 0.0. New best score: 0.692
Metric loss/va

### Best Checkpoint + Upload Artifacts to Weights and Biases

In [16]:
trainer.validate(model, dataloaders=valid_loader, ckpt_path="best")
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

run.summary.update({
    "accuracy/majority": np.bincount(y_test).max() / len(y_test),
    "best/loss_valid": checkpoint.best_model_score.item(),
    "best/checkpoint": Path(checkpoint.best_model_path).name,
    "early_stopping/stopped_epoch": early_stopping.stopped_epoch,
    "oov/tokens": embed_stats["tokens"],
    "oov/occurrences": embed_stats["oov"],
    "oov/rate": embed_stats["oov"] / embed_stats["tokens"],
    "oov/unique_types": len(oov_counts),
    "oov/empty_sentences": embed_stats["empty_sentences"],
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights/best-epoch=76.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights/best-epoch=76.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.6309999823570251     │
│        loss/valid         │    0.6427510380744934     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights/best-epoch=76.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights/best-epoch=76.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.4993472695350647     │
│         loss/test         │    0.8218773603439331     │
└───────────────────────────┴───────────────────────────┘

In [17]:
artifact = wandb.Artifact("glove-concatenate-lstm-unidirectional", type="model", metadata={"loss/valid": checkpoint.best_model_score.item()})
artifact.add_file(checkpoint.best_model_path)
run.log_artifact(artifact, aliases=["best"])

<Artifact glove-concatenate-lstm-unidirectional>

### Probability per Hypothesis and Answer

In [18]:
# best checkpoint, softmax over the 2 logits -> one probability distribution per test example
probabilities = torch.cat(trainer.predict(model, dataloaders=test_loader, ckpt_path="best")).numpy()

results = test.to_pandas()[TEXT_FIELDS + ["label"]]  # test_loader is not shuffled, same order as test
results["p_hypothesis_1"] = probabilities[:, 0]
results["p_hypothesis_2"] = probabilities[:, 1]
results["answer"] = probabilities.argmax(axis=1) + 1  # same numbering as label: 1 -> hypothesis_1, 2 -> hypothesis_2

print(f"accuracy from the answers: {(results['answer'] == results['label']).mean():.4f}")
results.head(10)

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights/best-epoch=76.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_concatenate_lstm_unidirectional/weights/best-epoch=76.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


accuracy from the answers: 0.4993


,observation_1,observation_2,hypothesis_1,hypothesis_2,label,p_hypothesis_1,p_hypothesis_2,answer
0,A man drove while listening to the radio.,The man called the radio station to ask.,The man heard a song he liked but the radio st...,He wanted to know the name of a song.,2,0.278918,0.721082,2
1,Ada was emptying the trash one night.,Ada had to pick up all the trash from the floor.,the bag broke as Ada was walking out the door.,The trash bag ripped open from a hole in the top.,1,0.443159,0.556841,2
2,Alex was home alone on a Friday night.,"Three friends were there to visit him, and the...",Alex was sad to spend the whole evening alone.,"Alex,posted on facebook that he was lonely at ...",2,0.239838,0.760162,2
3,Amber always thought she was the best tennis p...,Amber was disappointed that she lost to a new ...,amber is tennis player so good for playing but...,She be saded against a new player.,1,0.187937,0.812063,2
4,Amy decided to move from Wisconsin to Florida.,"However, she discovered Alaska was a lot of fun.",Amy did not think Alaska would be and decided ...,Amy did not think Alaska would be fun but deci...,2,0.497404,0.502596,2
5,Ann loved cake.,Ann came inside and watched as cakes were bein...,The local baker let Ann come into the workshop...,She passed by a meat shop one day.,1,0.716428,0.283572,1
6,Anna was jogging with her phone in her pocket.,She was so relieved to see it wasn't cracked!,Anne dropped her phone.,Anna's phone fell out of her pocket into nearb...,1,0.677779,0.322221,1
7,Beth was a model in a fashion show at her school.,"Beth missed the show, and was unable to tell a...",Beth passed out from overeating.,Beth went out and broke her phone with no ride...,2,0.517688,0.482312,1
8,Beth's brother Jake came to her house to check...,Jake slept on the couch to keep her safe.,Jake couldn't find the source of the noise so ...,Jake was scared to go to sleep alone in the ho...,1,0.363593,0.636407,2
9,Bob decided to hike in the jungles of Indonesia.,Bob had also forgotten his shoes.,Bob stopped in the middle of the hike because ...,Bob stopped in the middle of his hike to tie h...,1,0.578490,0.421510,1


In [19]:
wandb.finish()

accuracy/test,▁
accuracy/train,▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████
accuracy/valid,▁▁▂▂▃▃▂▃▄▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█▇▇▇▇███▇█████
epoch,▁▁▁▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
loss/test,▁
loss/train,█████████▇▆▆▆▅▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
loss/valid,██████████▇▇▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇████
accuracy/majority,0.50979
accuracy/test,0.49935
accuracy/train,0.66038
